In [24]:
!git clone https://github.com/MohamedElsayed75/FRW1NB1.git
%cd FRW1NB1/work/notebooks

Cloning into 'FRW1NB1'...
remote: Enumerating objects: 119, done.
remote: Counting objects: 100% (119/119), done.
remote: Compressing objects: 100% (90/90), done.
remote: Total 119 (delta 34), reused 79 (delta 13), pack-reused 0 (from 0)
Receiving objects: 100% (119/119), 1.86 MiB | 11.20 MiB/s, done.
Resolving deltas: 100% (34/34), done.
/content/FRW1NB1/work/notebooks/FRW1NB1/work/notebooks


In [25]:
!pip -q install duckdb huggingface_hub fsspec

In [26]:
import os
import duckdb
from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN")

if not HF_TOKEN:
    raise ValueError("HF_TOKEN was not found. Add it in Colab → Secrets.")

print("HF token found.")

HF token found.


In [27]:
con = duckdb.connect()

con.execute(f"""
INSTALL httpfs;
LOAD httpfs;

CREATE OR REPLACE SECRET hf (
    TYPE HUGGINGFACE,
    TOKEN '{HF_TOKEN}'
);
""")

print("DuckDB/Hugging Face connection ready.")

DuckDB/Hugging Face connection ready.


In [28]:
TABLE = "hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/**/*.parquet"

print(TABLE)

hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/**/*.parquet


# ML-07 — Baseline Action Score and Top-20 Review

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

## 1. Signal checks

My baseline is for the Refresh / Content Opportunity lane. I will focus on two signals:

1. Month-over-month GSC click change — a volume/decline signal.
2. Month-over-month GSC impression change — a second demand/visibility signal.

The first signal is directly related to FlyRank's quick-win/volume reasoning: pages with meaningful search volume are more actionable than pages with almost no demand.

Both signals are calculated from information available by the end of the observation period and do not use a future label.

In [29]:
import pandas as pd
import numpy as np

monthly = con.execute(f"""
SELECT
    client_hash_id,
    content_hash_id,
    month,

    AVG(gsc_clicks) AS avg_gsc_clicks,
    AVG(gsc_impressions) AS avg_gsc_impressions,
    AVG(gsc_avg_position) AS avg_position

FROM read_parquet('{TABLE}')

WHERE month IN ('2026-02', '2026-03')

GROUP BY
    client_hash_id,
    content_hash_id,
    month
""").fetchdf()

print("Rows loaded:", len(monthly))
monthly.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Rows loaded: 652983


,client_hash_id,content_hash_id,month,avg_gsc_clicks,avg_gsc_impressions,avg_position
0,client_e547b89c05043229,content_1eea820697c3b95a,2026-02,0.000000,10.678571,12.946228
1,client_e547b89c05043229,content_ccbb253f142217c3,2026-02,0.250000,57.071429,17.806923
2,client_e547b89c05043229,content_ae16a6b9cf64c80a,2026-02,0.000000,30.750000,7.852945
3,client_e547b89c05043229,content_9abd8b303f805847,2026-02,0.214286,26.178571,6.495085
4,client_e547b89c05043229,content_5f58c55cbfee172a,2026-02,0.000000,18.357143,10.490023


In [30]:
pivot = monthly.pivot_table(
    index=["client_hash_id", "content_hash_id"],
    columns="month",
    values=[
        "avg_gsc_clicks",
        "avg_gsc_impressions",
        "avg_position"
    ],
    aggfunc="first"
)

pivot.columns = [
    f"{metric}_{month}"
    for metric, month in pivot.columns
]

pivot = pivot.reset_index()

print("Content/client pairs:", len(pivot))
pivot.head()

Content/client pairs: 349411


,client_hash_id,content_hash_id,avg_gsc_clicks_2026-02,avg_gsc_clicks_2026-03,avg_gsc_impressions_2026-02,avg_gsc_impressions_2026-03,avg_position_2026-02,avg_position_2026-03
0,client_0797ff3a1fc9a6a5,content_004e9c4c32e88631,0.0,0.0,0.0,0.000000,NaN,NaN
1,client_0797ff3a1fc9a6a5,content_0236ef736698e17c,0.0,0.0,0.0,0.000000,NaN,NaN
2,client_0797ff3a1fc9a6a5,content_025f6cfd3c298870,0.0,0.0,0.0,0.000000,NaN,NaN
3,client_0797ff3a1fc9a6a5,content_0263d5f9b7a2ecd4,0.0,0.0,0.0,0.032258,NaN,9.0
4,client_0797ff3a1fc9a6a5,content_02752c6c1c60161f,0.0,0.0,0.0,0.000000,NaN,NaN


In [31]:
baseline = pivot.copy()

baseline["click_change_pct"] = np.where(
    baseline["avg_gsc_clicks_2026-02"] > 0,
    (
        baseline["avg_gsc_clicks_2026-03"]
        - baseline["avg_gsc_clicks_2026-02"]
    )
    / baseline["avg_gsc_clicks_2026-02"]
    * 100,
    np.nan
)

baseline["impression_change_pct"] = np.where(
    baseline["avg_gsc_impressions_2026-02"] > 0,
    (
        baseline["avg_gsc_impressions_2026-03"]
        - baseline["avg_gsc_impressions_2026-02"]
    )
    / baseline["avg_gsc_impressions_2026-02"]
    * 100,
    np.nan
)

baseline.head()

,client_hash_id,content_hash_id,avg_gsc_clicks_2026-02,avg_gsc_clicks_2026-03,avg_gsc_impressions_2026-02,avg_gsc_impressions_2026-03,avg_position_2026-02,avg_position_2026-03,click_change_pct,impression_change_pct
0,client_0797ff3a1fc9a6a5,content_004e9c4c32e88631,0.0,0.0,0.0,0.000000,NaN,NaN,NaN,NaN
1,client_0797ff3a1fc9a6a5,content_0236ef736698e17c,0.0,0.0,0.0,0.000000,NaN,NaN,NaN,NaN
2,client_0797ff3a1fc9a6a5,content_025f6cfd3c298870,0.0,0.0,0.0,0.000000,NaN,NaN,NaN,NaN
3,client_0797ff3a1fc9a6a5,content_0263d5f9b7a2ecd4,0.0,0.0,0.0,0.032258,NaN,9.0,NaN,NaN
4,client_0797ff3a1fc9a6a5,content_02752c6c1c60161f,0.0,0.0,0.0,0.000000,NaN,NaN,NaN,NaN


### Signal 1 — GSC click change

**Signal:** month-over-month change in average GSC clicks.

**Why I chose it:** clicks are a direct measure of organic search traffic volume. A meaningful decline is a plausible Refresh signal, while very low-volume pages are less actionable.

**Verdict:** CONFIRMED — the signal has enough variation across the March cohort to distinguish declining, stable, and growing pages.

In [32]:
baseline["click_bucket"] = pd.cut(
    baseline["click_change_pct"],
    bins=[-np.inf, -20, 0, 20, np.inf],
    labels=[
        "Strong decline (< -20%)",
        "Mild decline (-20% to 0%)",
        "Stable (0% to 20%)",
        "Growth (> 20%)"
    ]
)

click_bucket_table = (
    baseline["click_bucket"]
    .value_counts(dropna=False)
    .rename_axis("click_change_bucket")
    .reset_index(name="n")
)

click_bucket_table

,click_change_bucket,n
0,NaN,295673
1,Strong decline (< -20%),26205
2,Growth (> 20%),15847
3,Mild decline (-20% to 0%),8642
4,Stable (0% to 20%),3044


### Signal 2 — GSC impression change

**Signal:** month-over-month change in average GSC impressions.

**Why I chose it:** impressions measure search visibility. A fall in impressions can indicate that a page is losing search demand/visibility and therefore supports a Refresh interpretation.

**FlyRank connection:** this is related to the session's volume/quick-win reasoning because search visibility provides evidence that a page has an observable search footprint.

**Verdict:** CONFIRMED — impression change provides a useful second view of whether a page's search visibility is weakening.

In [33]:
baseline["impression_bucket"] = pd.cut(
    baseline["impression_change_pct"],
    bins=[-np.inf, -20, 0, 20, np.inf],
    labels=[
        "Strong decline (< -20%)",
        "Mild decline (-20% to 0%)",
        "Stable (0% to 20%)",
        "Growth (> 20%)"
    ]
)

impression_bucket_table = (
    baseline["impression_bucket"]
    .value_counts(dropna=False)
    .rename_axis("impression_change_bucket")
    .reset_index(name="n")
)

impression_bucket_table

,impression_change_bucket,n
0,NaN,204132
1,Growth (> 20%),59679
2,Strong decline (< -20%),53486
3,Mild decline (-20% to 0%),17008
4,Stable (0% to 20%),15106


In [34]:
baseline["has_search_volume"] = (
    baseline["avg_gsc_impressions_2026-03"] >= 100
)

In [35]:
print(
    "Rows with at least 100 March impressions:",
    baseline["has_search_volume"].sum()
)

print(
    "Rows below 100 March impressions:",
    (~baseline["has_search_volume"]).sum()
)

Rows with at least 100 March impressions: 21847
Rows below 100 March impressions: 327564


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

## 2. Baseline rule

I use one simple Refresh rule based on the two audited signals.

A content item receives a higher score when:

- GSC clicks declined materially month-over-month.
- GSC impressions also declined month-over-month.
- The page still has meaningful search visibility, so the opportunity is actionable.

The score is deliberately simple and interpretable. It is a baseline rather than a learned model.

Each row receives one reason code and one action label.

In [36]:
baseline["score"] = 0

# Click decline
baseline.loc[
    baseline["click_change_pct"] <= -20,
    "score"
] += 40

baseline.loc[
    (baseline["click_change_pct"] > -20) &
    (baseline["click_change_pct"] < 0),
    "score"
] += 20

# Impression decline
baseline.loc[
    baseline["impression_change_pct"] <= -20,
    "score"
] += 40

baseline.loc[
    (baseline["impression_change_pct"] > -20) &
    (baseline["impression_change_pct"] < 0),
    "score"
] += 20

# Meaningful search footprint
baseline.loc[
    baseline["has_search_volume"],
    "score"
] += 20

baseline["score"] = baseline["score"].astype(int)

baseline["score"].value_counts().sort_index()

,count
score,
0,251501
20,23348
40,53575
60,9099
80,10398
100,1490


In [37]:
def reason_code(row):
    click_decline = row["click_change_pct"]
    impression_decline = row["impression_change_pct"]

    if (
        pd.notna(click_decline)
        and pd.notna(impression_decline)
        and click_decline <= -20
        and impression_decline <= -20
    ):
        return "DOUBLE_DECLINE"

    if pd.notna(click_decline) and click_decline <= -20:
        return "CLICK_DECLINE"

    if pd.notna(impression_decline) and impression_decline <= -20:
        return "IMPRESSION_DECLINE"

    if pd.notna(click_decline) and click_decline < 0:
        return "MILD_CLICK_DECLINE"

    if pd.notna(impression_decline) and impression_decline < 0:
        return "MILD_IMPRESSION_DECLINE"

    return "NO_CLEAR_DECLINE"


baseline["reason_code"] = baseline.apply(
    reason_code,
    axis=1
)

In [38]:
def action_label(score):
    if score >= 80:
        return "REFRESH_NOW"
    elif score >= 60:
        return "REVIEW"
    elif score >= 40:
        return "MONITOR"
    else:
        return "NO_ACTION"

baseline["action"] = baseline["score"].apply(action_label)

### Ranked action queue

The queue is sorted from highest to lowest baseline score. Ties are broken by the largest click decline so that the most severe traffic losses appear first.

In [39]:
queue = baseline.sort_values(
    by=[
        "score",
        "click_change_pct",
        "impression_change_pct"
    ],
    ascending=[
        False,
        True,
        True
    ]
).reset_index(drop=True)

queue["rank"] = np.arange(1, len(queue) + 1)

queue[
    [
        "rank",
        "client_hash_id",
        "content_hash_id",
        "score",
        "reason_code",
        "action",
        "click_change_pct",
        "impression_change_pct",
        "avg_gsc_impressions_2026-03"
    ]
].head(20)

,rank,client_hash_id,content_hash_id,score,reason_code,action,click_change_pct,impression_change_pct,avg_gsc_impressions_2026-03
0,1,client_23a62021009f63c4,content_9345c24d1dbda27d,100,DOUBLE_DECLINE,REFRESH_NOW,-100.0,-73.086891,106.354839
1,2,client_73cda7b4e4f265ea,content_67dc193b282b586d,100,DOUBLE_DECLINE,REFRESH_NOW,-100.0,-68.418532,243.064516
2,3,client_73cda7b4e4f265ea,content_92d2c3fc8bf462c9,100,DOUBLE_DECLINE,REFRESH_NOW,-100.0,-56.469222,269.129032
3,4,client_23a62021009f63c4,content_85cd3ca3d410603a,100,DOUBLE_DECLINE,REFRESH_NOW,-100.0,-53.905742,131.483871
4,5,client_62f4a7e64f5e0096,content_155bbd621ff9af82,100,DOUBLE_DECLINE,REFRESH_NOW,-100.0,-50.938188,330.483871
5,6,client_e547b89c05043229,content_6a408c278519bbed,100,DOUBLE_DECLINE,REFRESH_NOW,-100.0,-49.425441,139.387097
6,7,client_3f0ce4d44fe94f3d,content_173b3082e7debaea,100,DOUBLE_DECLINE,REFRESH_NOW,-100.0,-46.203423,104.741935
7,8,client_62f4a7e64f5e0096,content_f26502a037991647,100,DOUBLE_DECLINE,REFRESH_NOW,-100.0,-45.301495,114.612903
8,9,client_23a62021009f63c4,content_cd5a15421cd60dff,100,DOUBLE_DECLINE,REFRESH_NOW,-100.0,-43.522051,285.354839
9,10,client_23a62021009f63c4,content_178c4ddfbd362148,100,DOUBLE_DECLINE,REFRESH_NOW,-100.0,-41.749674,114.129032


In [40]:
import os

os.makedirs("work/outputs", exist_ok=True)

output_columns = [
    "rank",
    "client_hash_id",
    "content_hash_id",
    "score",
    "reason_code",
    "action",
    "click_change_pct",
    "impression_change_pct",
    "avg_gsc_impressions_2026-03"
]

queue[output_columns].to_csv(
    "work/outputs/baseline_action_score.csv",
    index=False
)

print("CSV written:")
print("work/outputs/baseline_action_score.csv")

CSV written:
work/outputs/baseline_action_score.csv


In [41]:
pd.read_csv(
    "work/outputs/baseline_action_score.csv"
).head(10)

,rank,client_hash_id,content_hash_id,score,reason_code,action,click_change_pct,impression_change_pct,avg_gsc_impressions_2026-03
0,1,client_23a62021009f63c4,content_9345c24d1dbda27d,100,DOUBLE_DECLINE,REFRESH_NOW,-100.0,-73.086891,106.354839
1,2,client_73cda7b4e4f265ea,content_67dc193b282b586d,100,DOUBLE_DECLINE,REFRESH_NOW,-100.0,-68.418532,243.064516
2,3,client_73cda7b4e4f265ea,content_92d2c3fc8bf462c9,100,DOUBLE_DECLINE,REFRESH_NOW,-100.0,-56.469222,269.129032
3,4,client_23a62021009f63c4,content_85cd3ca3d410603a,100,DOUBLE_DECLINE,REFRESH_NOW,-100.0,-53.905742,131.483871
4,5,client_62f4a7e64f5e0096,content_155bbd621ff9af82,100,DOUBLE_DECLINE,REFRESH_NOW,-100.0,-50.938188,330.483871
5,6,client_e547b89c05043229,content_6a408c278519bbed,100,DOUBLE_DECLINE,REFRESH_NOW,-100.0,-49.425441,139.387097
6,7,client_3f0ce4d44fe94f3d,content_173b3082e7debaea,100,DOUBLE_DECLINE,REFRESH_NOW,-100.0,-46.203423,104.741935
7,8,client_62f4a7e64f5e0096,content_f26502a037991647,100,DOUBLE_DECLINE,REFRESH_NOW,-100.0,-45.301495,114.612903
8,9,client_23a62021009f63c4,content_cd5a15421cd60dff,100,DOUBLE_DECLINE,REFRESH_NOW,-100.0,-43.522051,285.354839
9,10,client_23a62021009f63c4,content_178c4ddfbd362148,100,DOUBLE_DECLINE,REFRESH_NOW,-100.0,-41.749674,114.129032


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

## 3. Top-10 review

I review the highest-ranked ten rows rather than treating the baseline score as ground truth. For each row I record the action, the evidence behind the ranking, and a condition that could make the recommendation wrong.

In [42]:
top10 = queue.head(10).copy()

top10[
    [
        "rank",
        "client_hash_id",
        "content_hash_id",
        "score",
        "reason_code",
        "action",
        "click_change_pct",
        "impression_change_pct"
    ]
]

,rank,client_hash_id,content_hash_id,score,reason_code,action,click_change_pct,impression_change_pct
0,1,client_23a62021009f63c4,content_9345c24d1dbda27d,100,DOUBLE_DECLINE,REFRESH_NOW,-100.0,-73.086891
1,2,client_73cda7b4e4f265ea,content_67dc193b282b586d,100,DOUBLE_DECLINE,REFRESH_NOW,-100.0,-68.418532
2,3,client_73cda7b4e4f265ea,content_92d2c3fc8bf462c9,100,DOUBLE_DECLINE,REFRESH_NOW,-100.0,-56.469222
3,4,client_23a62021009f63c4,content_85cd3ca3d410603a,100,DOUBLE_DECLINE,REFRESH_NOW,-100.0,-53.905742
4,5,client_62f4a7e64f5e0096,content_155bbd621ff9af82,100,DOUBLE_DECLINE,REFRESH_NOW,-100.0,-50.938188
5,6,client_e547b89c05043229,content_6a408c278519bbed,100,DOUBLE_DECLINE,REFRESH_NOW,-100.0,-49.425441
6,7,client_3f0ce4d44fe94f3d,content_173b3082e7debaea,100,DOUBLE_DECLINE,REFRESH_NOW,-100.0,-46.203423
7,8,client_62f4a7e64f5e0096,content_f26502a037991647,100,DOUBLE_DECLINE,REFRESH_NOW,-100.0,-45.301495
8,9,client_23a62021009f63c4,content_cd5a15421cd60dff,100,DOUBLE_DECLINE,REFRESH_NOW,-100.0,-43.522051
9,10,client_23a62021009f63c4,content_178c4ddfbd362148,100,DOUBLE_DECLINE,REFRESH_NOW,-100.0,-41.749674


In [43]:
def why_here(row):
    return (
        f"Score {row['score']}: "
        f"clicks changed {row['click_change_pct']:.1f}% and "
        f"impressions changed {row['impression_change_pct']:.1f}%; "
        f"reason={row['reason_code']}."
    )

def what_makes_wrong(row):
    if row["reason_code"] == "DOUBLE_DECLINE":
        return (
            "The decline could reflect seasonality, a temporary search-demand "
            "change, tracking changes, or a site-wide issue rather than content quality."
        )

    if row["reason_code"] == "CLICK_DECLINE":
        return (
            "Clicks may have fallen because of SERP changes, lower CTR, "
            "seasonality, or a change in search demand rather than stale content."
        )

    if row["reason_code"] == "IMPRESSION_DECLINE":
        return (
            "Lower impressions may reflect search-demand changes or indexing/ranking "
            "changes rather than a page that needs refreshing."
        )

    return (
        "The simple month-over-month signal may be noisy and may not indicate "
        "a genuine content opportunity."
    )

top10["why_it_is_here"] = top10.apply(
    why_here,
    axis=1
)

top10["what_would_make_it_wrong"] = top10.apply(
    what_makes_wrong,
    axis=1
)

top10[
    [
        "rank",
        "action",
        "reason_code",
        "why_it_is_here",
        "what_would_make_it_wrong"
    ]
]

,rank,action,reason_code,why_it_is_here,what_would_make_it_wrong
0,1,REFRESH_NOW,DOUBLE_DECLINE,Score 100: clicks changed -100.0% and impressi...,"The decline could reflect seasonality, a tempo..."
1,2,REFRESH_NOW,DOUBLE_DECLINE,Score 100: clicks changed -100.0% and impressi...,"The decline could reflect seasonality, a tempo..."
2,3,REFRESH_NOW,DOUBLE_DECLINE,Score 100: clicks changed -100.0% and impressi...,"The decline could reflect seasonality, a tempo..."
3,4,REFRESH_NOW,DOUBLE_DECLINE,Score 100: clicks changed -100.0% and impressi...,"The decline could reflect seasonality, a tempo..."
4,5,REFRESH_NOW,DOUBLE_DECLINE,Score 100: clicks changed -100.0% and impressi...,"The decline could reflect seasonality, a tempo..."
5,6,REFRESH_NOW,DOUBLE_DECLINE,Score 100: clicks changed -100.0% and impressi...,"The decline could reflect seasonality, a tempo..."
6,7,REFRESH_NOW,DOUBLE_DECLINE,Score 100: clicks changed -100.0% and impressi...,"The decline could reflect seasonality, a tempo..."
7,8,REFRESH_NOW,DOUBLE_DECLINE,Score 100: clicks changed -100.0% and impressi...,"The decline could reflect seasonality, a tempo..."
8,9,REFRESH_NOW,DOUBLE_DECLINE,Score 100: clicks changed -100.0% and impressi...,"The decline could reflect seasonality, a tempo..."
9,10,REFRESH_NOW,DOUBLE_DECLINE,Score 100: clicks changed -100.0% and impressi...,"The decline could reflect seasonality, a tempo..."


In [44]:
for _, row in top10.iterrows():
    print(
        f"{int(row['rank'])}. "
        f"{row['action']} — "
        f"{row['why_it_is_here']} "
        f"What could make it wrong: "
        f"{row['what_would_make_it_wrong']}"
    )

1. REFRESH_NOW — Score 100: clicks changed -100.0% and impressions changed -73.1%; reason=DOUBLE_DECLINE. What could make it wrong: The decline could reflect seasonality, a temporary search-demand change, tracking changes, or a site-wide issue rather than content quality.
2. REFRESH_NOW — Score 100: clicks changed -100.0% and impressions changed -68.4%; reason=DOUBLE_DECLINE. What could make it wrong: The decline could reflect seasonality, a temporary search-demand change, tracking changes, or a site-wide issue rather than content quality.
3. REFRESH_NOW — Score 100: clicks changed -100.0% and impressions changed -56.5%; reason=DOUBLE_DECLINE. What could make it wrong: The decline could reflect seasonality, a temporary search-demand change, tracking changes, or a site-wide issue rather than content quality.
4. REFRESH_NOW — Score 100: clicks changed -100.0% and impressions changed -53.9%; reason=DOUBLE_DECLINE. What could make it wrong: The decline could reflect seasonality, a temporar

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

## 4. Weak picks

A weak pick is a row that receives a relatively high score from the simple rule but where the evidence is easy to misinterpret. These are useful because the baseline is intentionally simple and should not be treated as ground truth.

In [45]:
weak_picks = queue[
    (queue["score"] >= 60) &
    (
        (queue["click_change_pct"].abs() < 10) |
        (queue["impression_change_pct"].abs() < 10)
    )
].head(5)

weak_picks[
    [
        "rank",
        "score",
        "reason_code",
        "action",
        "click_change_pct",
        "impression_change_pct"
    ]
]

,rank,score,reason_code,action,click_change_pct,impression_change_pct
6423,6424,80,CLICK_DECLINE,REFRESH_NOW,-100.0,-9.934958
6424,6425,80,CLICK_DECLINE,REFRESH_NOW,-100.0,-9.593085
6425,6426,80,CLICK_DECLINE,REFRESH_NOW,-100.0,-8.674790
6426,6427,80,CLICK_DECLINE,REFRESH_NOW,-100.0,-7.510915
6427,6428,80,CLICK_DECLINE,REFRESH_NOW,-100.0,-5.577297


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.

In [46]:
print("SELF-CHECK")
print("===========")

checks = {
    "Two signal checks completed": True,
    "Signal 1 bucket table has n": "n" in click_bucket_table.columns,
    "Signal 2 bucket table has n": "n" in impression_bucket_table.columns,
    "Signal 1 verdict provided": True,
    "Signal 2 verdict provided": True,
    "One baseline score created": "score" in queue.columns,
    "One reason code created": "reason_code" in queue.columns,
    "Action label created": "action" in queue.columns,
    "Ranked queue created": "rank" in queue.columns,
    "CSV written": os.path.exists(
        "work/outputs/baseline_action_score.csv"
    ),
    "Top 10 reviewed": len(top10) == 10,
    "What-would-make-it-wrong included": (
        "what_would_make_it_wrong" in top10.columns
    ),
}

for name, passed in checks.items():
    print(f"[{'PASS' if passed else 'FAIL'}] {name}")

SELF-CHECK
[PASS] Two signal checks completed
[PASS] Signal 1 bucket table has n
[PASS] Signal 2 bucket table has n
[PASS] Signal 1 verdict provided
[PASS] Signal 2 verdict provided
[PASS] One baseline score created
[PASS] One reason code created
[PASS] Action label created
[PASS] Ranked queue created
[PASS] CSV written
[PASS] Top 10 reviewed
[PASS] What-would-make-it-wrong included
